# Fase 4 v2 — 04: homografía TEMPORAL estable (SAM3 ancla + flujo óptico)

Arregla el jitter ("las líneas se mueven cada frame") y la oclusión: **no re-detecta**
líneas cada frame. SAM3 ancla cuando hay detección confiable (alfombra `green_floor`
+ porterías `yellow_zone`/`blue_zone`); entre anclas, **propaga H siguiendo puntos
con flujo óptico Lucas-Kanade** → H se mueve solo con la cámara, sin re-detección.
Re-ancla solo si la cámara de verdad se movió (corrección suave EMA).

Se corre en **cenital Y lateral** (no cenital) para ver ambos. Salida: videos overlay
con el template pegado + estela de puntos seguidos + status por frame.

In [ ]:
import sys, os, json, shutil, subprocess
from pathlib import Path
import numpy as np
import cv2, decord

REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core import field_landmarks as fl
from src.core.homography_tracked import VideoHomographyTracked
from src.core.sam3_loader import load_sam3
from src.core.segmentation import detect_classes_in_frame, _load_classes
from src.core.minimap_pipeline import _largest_mask, _largest_component
from src.core.homography import mask_centroid

OUT = REPO / 'notebooks/fase_4_homografia/v2_robust/outputs/videos'
OUT.mkdir(parents=True, exist_ok=True)
bundle = load_sam3()
ALLCLS = _load_classes()
FIELD_CLS = [c for c in ALLCLS if c['name'] in ('green_floor','yellow_zone','blue_zone')]
print('REPO:', REPO, '| field classes:', [c['name'] for c in FIELD_CLS])

In [ ]:
def field_inputs(frame_rgb):
    """SAM3 -> (carpet_mask, yellow_centroid, blue_centroid)."""
    dets = detect_classes_in_frame(frame_rgb, FIELD_CLS, bundle)
    cm = _largest_mask(dets.get('green_floor', []))
    if cm is None:
        return None, None, None
    cm = _largest_component(cm)
    ym = _largest_mask(dets.get('yellow_zone', []))
    bm = _largest_mask(dets.get('blue_zone', []))
    yc = mask_centroid(ym) if ym is not None else None
    bc = mask_centroid(bm) if bm is not None else None
    return cm, yc, bc

def cm_to_img(Hinv, pts_cm):
    p = np.asarray(pts_cm, np.float64).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(p, Hinv).reshape(-1, 2)

CIRC = np.stack([fl.CENTER_CIRCLE[0]+fl.CENTER_CIRCLE[2]*np.cos(np.linspace(0,2*np.pi,60)),
                 fl.CENTER_CIRCLE[1]+fl.CENTER_CIRCLE[2]*np.sin(np.linspace(0,2*np.pi,60))],1)
STCOL = {'anchored':(0,255,0),'tracked':(0,220,255),'corrected':(0,255,0),
         'reanchored':(255,0,255),'lost':(0,0,255),'no_carpet':(0,0,255)}

def draw_overlay(frame_rgb, H, status, pts_img=None):
    img = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).copy()
    cv2.putText(img, status, (20,46), cv2.FONT_HERSHEY_SIMPLEX, 1.1, STCOL.get(status,(255,255,255)), 3, cv2.LINE_AA)
    if pts_img is not None:
        for p in pts_img:
            cv2.circle(img, (int(p[0]),int(p[1])), 2, (0,220,255), -1)
    if H is None:
        return img
    try:
        Hinv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return img
    rect = [fl.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    cv2.polylines(img, [cm_to_img(Hinv, rect).astype(np.int32)], True, (0,255,0), 2, cv2.LINE_AA)
    cl = cm_to_img(Hinv, [fl.LANDMARK_POINTS['center_top'], fl.LANDMARK_POINTS['center_bot']]).astype(np.int32)
    cv2.line(img, tuple(cl[0]), tuple(cl[1]), (0,255,255), 2, cv2.LINE_AA)
    cv2.polylines(img, [cm_to_img(Hinv, CIRC).astype(np.int32)], True, (255,128,0), 2, cv2.LINE_AA)
    return img

In [ ]:
CLIPS = {
    'cenital_9933': '/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV',
    'lateral_9913': '/workspace/Meta_Glasses/18abril/Cámaras/IMG_9913.MOV',
}
MAX_FRAMES, STEP, OUT_W, FPS = 100, 3, 960, 12

def transcode(src, dst):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(src),'-vcodec','libx264','-pix_fmt','yuv420p',str(dst)], check=True)

summary = {}
for name, path in CLIPS.items():
    vr = decord.VideoReader(path); n = len(vr)
    idxs = list(range(0, n, STEP))[:MAX_FRAMES]
    vh = VideoHomographyTracked()
    f0 = vr[idxs[0]].asnumpy(); H0, W0 = f0.shape[:2]
    sc = OUT_W / W0; oh = int(round(H0*sc))
    tmp = str(OUT / f'{name}_trk_tmp.mp4')
    wr = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (OUT_W, oh))
    for k, i in enumerate(idxs):
        rgb = vr[i].asnumpy()
        carpet, yc, bc = field_inputs(rgb)
        if carpet is None:
            H, status = vh.H, 'no_carpet'
        else:
            H, status = vh.update(rgb, carpet, yc, bc)
        ov = draw_overlay(rgb, H, status, vh.pts_img)
        wr.write(cv2.resize(ov, (OUT_W, oh)))
        if k % 20 == 0: print(f'  {name} {k}/{len(idxs)} {status}')
    wr.release()
    final = str(OUT / f'{name}_tracked.mp4')
    transcode(tmp, final); os.remove(tmp)
    summary[name] = {'frames': len(idxs), **vh.stats(), 'mb': round(os.path.getsize(final)/1e6,2)}
    print(name, summary[name])
json.dump(summary, open(OUT/'tracked_summary.json','w'), indent=2)
print('SUMMARY', summary)

In [ ]:
zb = str(OUT.parent / 'tracked_videos')
if os.path.exists(zb+'.zip'): os.remove(zb+'.zip')
shutil.make_archive(zb, 'zip', OUT, '.')
print('videos:', [p.name for p in OUT.glob('*_tracked.mp4')])

## Resumen

Status por frame: `anchored` (SAM3 fijó ancla), `tracked` (propagado por flujo óptico,
sin re-detección → estable), `corrected` (deriva corregida suave), `reanchored`
(cámara se movió de verdad), `lost`/`no_carpet` (sin alfombra → congela última H).
Esperado vs nb02/03: muchas menos rejected/jitter; las líneas **se quedan pegadas**.